In [26]:
# Preparation
import pandas as pd, numpy as np, sqlalchemy as sqla
import decouple, shutil, glob
import re
from datetime import datetime
from sqlalchemy import String
from IPython.display import display
 
pd.set_option("display.max_rows", None)
pd.set_option('display.max_columns', None)
pd.set_option("display.float_format", "{:.2f}".format)

mis_fldr_path = decouple.config("MIS_FILE_PATH")
engine = sqla.create_engine(
    decouple.config("MIS_DB"),
    connect_args={"charset": "utf8mb4"}
    )

In [27]:
# ── Load tables ────────────────────────────────────────────────────────────────
sout   = pd.read_sql("SELECT * FROM sales.sout_prtl", engine)
ref_ad = pd.read_sql("SELECT * FROM ref.account_details", engine)
ref_sd = pd.read_sql("SELECT * FROM ref.store_details", engine)
ref_l  = pd.read_sql("SELECT * FROM ref.lead_names", engine)
ref_ic = pd.read_sql("SELECT * FROM ref.item_codes", engine)
ref_pd = pd.read_sql("SELECT * FROM ref.product_details", engine)

# ── Filters and type fixes ─────────────────────────────────────────────────────
sout = sout[sout["year"] > 2024].copy()

sout["year"]         = sout["year"].astype(int)
ref_l["year"]        = ref_l["year"].astype(int)
sout["store_code"]   = sout["store_code"].astype(str)
ref_sd["store_code"] = ref_sd["store_code"].astype(str)
sout["item_code"]    = sout["item_code"].astype(str)
ref_ic["item_code"]  = ref_ic["item_code"].astype(str)

In [28]:
# ── Add account details (ref_ad) ───────────────────────────────────────────────
# Pull only needed columns to avoid bpc_region collision with ref_sd
ref_ad_cols = ref_ad[["account_name", "account_chain", "sales_group", "lead_id"]]
sout_df = pd.merge(sout, ref_ad_cols, on="account_name", how="left")

# ── Add store details (ref_sd) — bpc_region comes from HERE per SQL ────────────
sout_df = pd.merge(sout_df, ref_sd, on=["account_name", "store_code"], how="left")

# ── Add assigned team leader ───────────────────────────────────────────────────
sout_df = pd.merge(sout_df, ref_l, on=["lead_id", "year", "month"], how="left")

# ── Add item codes ─────────────────────────────────────────────────────────────
sout_df = pd.merge(sout_df, ref_ic, on=["account_chain", "item_code"], how="left")

# ── Add product details ────────────────────────────────────────────────────────
sout_df = pd.merge(sout_df, ref_pd, left_on="product_code", right_on="product_code1", how="left")

In [29]:
# ── Resolve any remaining duplicate columns ────────────────────────────────────
dup_bases = set(c.replace("_x", "").replace("_y", "")
                for c in sout_df.columns if c.endswith(("_x", "_y")))
for base in dup_bases:
    x_col, y_col = f"{base}_x", f"{base}_y"
    if x_col in sout_df.columns and y_col in sout_df.columns:
        sout_df[base] = sout_df[x_col].fillna(sout_df[y_col])
        sout_df = sout_df.drop(columns=[x_col, y_col])

# ── Select and reorder columns ─────────────────────────────────────────────────
sout_df = sout_df[[
    "year", "month", "sales_group", "account_name", "account_chain",
    "store_name", "store_code", "city", "province", "region", "bpc_region",
    "lead_name", "item_desc", "item_code", "product_code", "product_name",
    "brand", "packaging", "volume", "product_class", "usage", "product_type",
    "product_category", "target_type", "qty", "amount", "net_amount"
]]

sout_df.insert(2, "type", "Sell-Out")
sout_df = sout_df.sort_values("account_name").reset_index(drop=True)

sout_df.head()

,year,month,type,sales_group,account_name,account_chain,store_name,store_code,city,province,region,bpc_region,lead_name,item_desc,item_code,product_code,product_name,brand,packaging,volume,product_class,usage,product_type,product_category,target_type,qty,amount,net_amount
0,2025,January,Sell-Out,S2,Mercury Drug Corporation,Mercury Drug Corporation,Mercury Drug Corporation Quezon City Cubao Pin...,0012,Quezon City,Metro Manila,NCR,NCR,Vacant - MDC,None,312837,PFWS,iWhite Korea Glow-Up Whip Wash - 10ml (PFWS),iWhite Korea,Sachet,10ml,Skin Care,Daily,Facial Wash,Regular,Baseline,24.00,576.00,403.20
1,2025,December,Sell-Out,S2,Mercury Drug Corporation,Mercury Drug Corporation,Mercury Drug Corporation Kalookan - C-3,0240,Caloocan,Metro Manila,NCR,NCR,Rubylie Fabrigas,None,312877,BFWS,iWhite Korea Facial Wash Whitening Vita - 10ml...,iWhite Korea,Sachet,10ml,Skin Care,Daily,Facial Wash,Regular,Baseline,24.00,552.00,386.40
2,2025,December,Sell-Out,S2,Mercury Drug Corporation,Mercury Drug Corporation,Mercury Drug Corporation Kalookan - C-3,0240,Caloocan,Metro Manila,NCR,NCR,Rubylie Fabrigas,None,312861,FCT,iWhite Korea Facial Cream Whitening Vita - 65m...,iWhite Korea,Tube,65ml,Skin Care,Daily,Facial Cream,Regular,Baseline,2.00,450.00,315.00
3,2025,December,Sell-Out,S2,Mercury Drug Corporation,Mercury Drug Corporation,Mercury Drug Corporation Kalookan - C-3,0240,Caloocan,Metro Manila,NCR,NCR,Rubylie Fabrigas,None,312859,BACT,iWhite Korea Aqua Moisturizer Whitening Vita -...,iWhite Korea,Tube,50ml,Skin Care,Daily,Moisturizer,Regular,Baseline,1.00,224.00,156.80
4,2025,December,Sell-Out,S2,Mercury Drug Corporation,Mercury Drug Corporation,Mercury Drug Corporation Kalookan - C-3,0240,Caloocan,Metro Manila,NCR,NCR,Rubylie Fabrigas,None,312857,NPS,iWhite Korea The Original Nose Pack - 3.5ml (NPS),iWhite Korea,Sachet,3.5ml,Skin Care,Weekly,Pore Care,Regular,Baseline,24.00,480.00,336.00


In [30]:
# ── Write output CSV files ─────────────────────────────────────────────────────
sout_df.to_csv(rf"{mis_fldr_path}Report Templates\Generated Sell-Out YTD.csv", index=False, encoding="utf-8-sig")
# sout_df.to_csv(r"C:\eli_dump\Backup\Database Files\YTD Reports\Generated Sell-Out YTD.csv", index=False, encoding="utf-8-sig")